# Likelihood Analysis and Experiment Matching

This notebook demonstrates how to use DMRecoilLab to infer dark-matter parameters from recoil-energy data and compare the resulting signal predictions across different experiments.

We consider a simple scenario with a single observed recoil event and perform a likelihood scan in the plane of
$$(m_\chi,\delta)$$
where $m_\chi$​ is the WIMP mass and $\delta$ is the inelastic mass splitting.

The workflow consists of three steps:
- Likelihood evaluation
- Confidence-region visualization
- Experiment matching

Although only a single recoil event is used here, the same machinery applies to larger event samples and can be used for parameter estimation, detector comparisons, and phenomenological studies of elastic and inelastic dark-matter scattering.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.nuclei.isotopes import Xe131
from src.nuclei.form_factors import HelmFormFactor

## Experimental setup.

In this example we use a xenon target similar to LZ.

In [ ]:
from src.detectors.detectors import ResponseDetector
from src.utils.tabulated_data import TabulatedData

lz_acceptance = TabulatedData(
    x=[
        0.0, 1.0e3, 1.5e3, 2.8e3, 3.0e3,
        50e3, 250e3, 270e3, 300e3],
    y=[
        0.0, 0.0, 0.20, 0.80, 0.90,
        0.95, 0.95, 0.50, 0.00])

LZ2026 = ResponseDetector(nucleus=Xe131, mass_kg=2.84e3, exposure_days=365,
                          resolution_sigma=0.3e3, acceptance_curve=lz_acceptance)

## Likelihood scan in the $(m_\chi,\delta)$ plane

We consider a single reconstructed recoil event and evaluate the profile likelihood over a grid of WIMP masses and inelastic mass splittings.

For each grid point $(m_\chi,\delta)$, a spin-independent interaction model is constructed and the normalized recoil spectrum is evaluated for the specified detector configuration.
The likelihood is maximised with respect to the expected event count, so the scan is driven primarily by spectral shape information rather than the overall interaction normalization.

The resulting likelihood surface is stored as a LikelihoodGridVSMassDelta object, which provides convenience methods for extracting best-fit parameters and visualizing likelihood-ratio test statistics.

In [ ]:
from src.analysis.likelihoods import likelihood_grid_si_mass_delta

# WIMP masses and delta splitting energies are expressed in eV
wimp_masses = np.geomspace(2e10, 5e12, 10)
deltas = np.linspace(-1400e3, 400e3, 45)

# Recoil energies are expressed in eV
Ei = np.asarray([248.e3])

logL_grid = likelihood_grid_si_mass_delta(
    wimp_masses=wimp_masses,
    deltas=deltas,
    detector=LZ2026,
    energies=Ei,
    form_factor=HelmFormFactor(Xe131))

## Best-fit parameters

The likelihood grid stores the profile likelihood relative to its global maximum.

The corresponding test statistic is
$q = -2 \Delta \ln L$,
for which the best-fit point satisfies $q = 0$.

We extract and display the preferred WIMP mass and mass splitting.

In [ ]:
q = logL_grid.q_cap_25

m_best_GeV = logL_grid.best_fit_mass / 1e9
d_best_keV = logL_grid.best_fit_delta / 1e3

print("Best fit model:")
print(f"mX = {m_best_GeV:.1f} GeV")
print(f"delta = {d_best_keV:.1f} keV")

## Likelihood-ratio map

The following plot shows the likelihood-ratio test statistic
$$q=−2Δln⁡L$$
across the scanned parameter space.

Values close to zero indicate parameter combinations whose predicted recoil spectrum is most compatible with the observed event energy.
Larger values correspond to increasingly disfavored regions.

For visualization purposes the color scale is capped at $q=25$.

In [ ]:
from src.plotting.plots_likelihoods import plot_q_grid_mass_delta

plot_q_grid_mass_delta(grid=logL_grid)

## Confidence contours

Confidence contours are obtained from the likelihood-ratio statistic.
For two fitted parameters, the standard asymptotic confidence regions correspond to

C.L.  -> q

68.3% ->  2.3

95.4% ->  6.18

99.7% -> 11.83

These contours illustrate the range of masses and mass splittings that produce recoil spectra compatible with the observed event.

In [ ]:
from src.plotting.plots_likelihoods import plot_q_contour_mass_delta

plot_q_contour_mass_delta(grid=logL_grid)

## Event-count matching between experiments

The likelihood analysis constrains the shape of the recoil spectrum.
We can also compare the expected signal normalization between different detectors.

For each point in the $(m_\chi,\delta)$ plane, we determine the number of events that would be expected in DS-20K if the model were normalized to produce one event in LZ.

This calculation highlights how detector target material, exposure, and kinematic thresholds affect the relative signal yield.

In [ ]:
from src.detectors.reference_detectors import DS20K_10YEAR
from src.analysis.experiment_matching import matched_counts_grid_si

grid = matched_counts_grid_si(wimp_masses=wimp_masses,
                              deltas=deltas,
                              counts_a=1,
                              detector_a=LZ2026,
                              detector_b=DS20K_10YEAR)

## Matched-count map

The matched-count grid shows the number of expected events in the second experiment for models that would produce exactly one event in the reference experiment.
Regions with large matched counts correspond to scenarios where the second detector would have substantially greater sensitivity to the same underlying signal.

In [ ]:
from src.plotting.plots_experiment_matching import plot_matched_counts

plot_matched_counts(grid=grid)

## Combining spectral and rate information

The final figure overlays the likelihood contours from the recoil-energy measurement with the matched-count prediction for DS-20K.

This visualization combines two complementary pieces of information:
- the likelihood contours identify models compatible with the observed recoil energy;
- the matched-count map indicates how many events those same models would predict in a second experiment.

Together they provide a useful bridge between parameter inference and experimental forecasting.

In [ ]:
from matplotlib.colors import LogNorm

M, D = np.meshgrid(
    wimp_masses / 1e9,   # GeV
    deltas / 1e3,        # keV
    indexing="ij"
)

fig, ax = plt.subplots(figsize=(8,6))

pcm = ax.contourf(
    M,
    D,
    grid.counts_b_cap_1000,
    levels=40,
    norm=LogNorm()
)

cbar = plt.colorbar(
    pcm,
    ax=ax,
    label="DS-20k 10 years matched counts (cap to 1000)"
)

levels = [2.30, 6.18]

cs = ax.contour(
    M,
    D,
    q,
    levels=levels,
    colors=["orange", "red"],
    linewidths=2
)

ax.clabel(cs,
          fmt={2.30: "68%", 6.18: "95%"})

ax.plot(
    m_best_GeV,
    d_best_keV,
    marker="*",
    markersize=18,
    color="magenta",
    markeredgecolor="black",
)

ax.set_xscale("log")

ax.set_xlabel(r"$m_\chi$ [GeV]")
ax.set_ylabel(r"$\delta$ [keV]")

plt.show()